In [40]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
import time
import math as m

In [2]:
crime_schema = T.StructType([

T.StructField("DR_NO",T.IntegerType(),True),
T.StructField("Date Rptd",T.DateType(),True),
T.StructField("DATE OCC",T.DateType(),True),
T.StructField("TIME OCC",T.IntegerType(),True),
T.StructField("AREA",T.IntegerType(),True),
T.StructField("AREA NAME",T.StringType(),True),
T.StructField("Rpt Dist No",T.IntegerType(),True),
T.StructField("Part 1-2",T.IntegerType(),True),
T.StructField("Crm Cd",T.IntegerType(),True),
T.StructField("Crm Cd Desc",T.StringType(),True),
T.StructField("Mocodes",T.StringType(),True),
T.StructField("Vict Age",T.IntegerType(),True),
T.StructField("Vict Sex",T.StringType(),True),
T.StructField("Vict Descent",T.StringType(),True),
T.StructField("Premis Cd",T.DoubleType(),True),
T.StructField("Premis Desc",T.StringType(),True),
T.StructField("Weapon Used Cd",T.DoubleType(),True),
T.StructField("Weapon Desc",T.StringType(),True),
T.StructField("Status",T.StringType(),True),
T.StructField("Status Desc",T.StringType(),True),
T.StructField("Crm Cd 1",T.DoubleType(),True),
T.StructField("Crm Cd 2",T.DoubleType(),True),
T.StructField("Crm Cd 3",T.DoubleType(),True),
T.StructField("Crm Cd 4",T.DoubleType(),True),
T.StructField("LOCATION",T.StringType(),True),
T.StructField("Cross Street",T.StringType(),True),
T.StructField("LAT",T.DoubleType(),True),
T.StructField("LON",T.DoubleType(),True) 
])

In [3]:
spark = (SparkSession.builder
    .appName("CrimeData")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0,"
            "org.postgresql:postgresql:42.7.4")
    .getOrCreate())

In [4]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [17]:
df_csv = (spark.read
    .schema(crime_schema)
    .option("header", True)
    .option("dateFormat", "yyyy MMM dd hh:mm:ss a")
    .csv("Crime Data.csv"))
    
df_csv.printSchema()
print(df_csv.count())

root
 |-- DR_NO: integer (nullable = true)
 |-- Date Rptd: date (nullable = true)
 |-- DATE OCC: date (nullable = true)
 |-- TIME OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA NAME: string (nullable = true)
 |-- Rpt Dist No: integer (nullable = true)
 |-- Part 1-2: integer (nullable = true)
 |-- Crm Cd: integer (nullable = true)
 |-- Crm Cd Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict Age: integer (nullable = true)
 |-- Vict Sex: string (nullable = true)
 |-- Vict Descent: string (nullable = true)
 |-- Premis Cd: double (nullable = true)
 |-- Premis Desc: string (nullable = true)
 |-- Weapon Used Cd: double (nullable = true)
 |-- Weapon Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status Desc: string (nullable = true)
 |-- Crm Cd 1: double (nullable = true)
 |-- Crm Cd 2: double (nullable = true)
 |-- Crm Cd 3: double (nullable = true)
 |-- Crm Cd 4: double (nullable = true)
 |-- LOCATION: s

In [44]:
df_csv.write.mode("overwrite").parquet("Crime Data.parquet")
df_pq = (spark.read
    .parquet("Crime Data.parquet"))
    
df_pq.printSchema()
print(df_pq.count())

root
 |-- DR_NO: integer (nullable = true)
 |-- Date Rptd: date (nullable = true)
 |-- DATE OCC: date (nullable = true)
 |-- TIME OCC: integer (nullable = true)
 |-- AREA: integer (nullable = true)
 |-- AREA NAME: string (nullable = true)
 |-- Rpt Dist No: integer (nullable = true)
 |-- Part 1-2: integer (nullable = true)
 |-- Crm Cd: integer (nullable = true)
 |-- Crm Cd Desc: string (nullable = true)
 |-- Mocodes: string (nullable = true)
 |-- Vict Age: integer (nullable = true)
 |-- Vict Sex: string (nullable = true)
 |-- Vict Descent: string (nullable = true)
 |-- Premis Cd: double (nullable = true)
 |-- Premis Desc: string (nullable = true)
 |-- Weapon Used Cd: double (nullable = true)
 |-- Weapon Desc: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Status Desc: string (nullable = true)
 |-- Crm Cd 1: double (nullable = true)
 |-- Crm Cd 2: double (nullable = true)
 |-- Crm Cd 3: double (nullable = true)
 |-- Crm Cd 4: double (nullable = true)
 |-- LOCATION: s

In [49]:
#1.	What are the ten most common recorded crime types in this dataset?

start_pq = time.time()
result1_pq = spark.sql("""
SELECT `Crm Cd`,`Crm Cd Desc`, COUNT(*) AS `Count of Cases`
FROM {crime_df}
WHERE `Crm Cd` IS NOT NULL
GROUP BY `Crm Cd`,`Crm Cd Desc`
ORDER BY `Count of Cases` DESC
LIMIT 10
""", crime_df=df_pq)
result1_pq.collect()
end_pq = time.time()
diff_pq = end_pq-start_pq

start_csv = time.time()
result1_csv = spark.sql("""
SELECT `Crm Cd`,`Crm Cd Desc`, COUNT(*) AS `Count of Cases`
FROM {crime_df}
WHERE `Crm Cd` IS NOT NULL
GROUP BY `Crm Cd`, `Crm Cd Desc`
ORDER BY `Count of Cases` DESC
LIMIT 10
""", crime_df=df_csv)
result1_csv.collect()
end_csv = time.time()
diff_csv = end_csv-start_csv
print("Query Analytics for CSV vs Parquet Comparison\n")
print("CSV:",round(diff_csv,2),"\n")
print("Parquet:",round(diff_pq,2),"\n")
if diff_pq < diff_csv:
    print("Parquet is faster by:",round(diff_csv-diff_pq,2),"seconds\n")
else:
    print("CSV is faster by:",round(diff_pq-diff_csv,2),"seconds\n")
print("What are the ten most common recorded crime types in this dataset?\n")
result1_csv

Query Analytics for CSV vs Parquet Comparison

CSV: 2.38 

Parquet: 1.51 

Parquet is faster by: 0.87 seconds

What are the ten most common recorded crime types in this dataset?



Crm Cd,Crm Cd Desc,Count of Cases
510,VEHICLE - STOLEN,115184
624,BATTERY - SIMPLE ...,74821
330,BURGLARY FROM VEH...,63515
354,THEFT OF IDENTITY,62536
740,VANDALISM - FELON...,61086
310,BURGLARY,57871
440,THEFT PLAIN - PET...,53716
230,ASSAULT WITH DEAD...,53523
626,INTIMATE PARTNER ...,46712
420,THEFT FROM MOTOR ...,41311


In [45]:
#2.	Which five LAPD areas have the most recorded incidents in this dataset?

start_pq = time.time()
result2_pq = spark.sql("""
SELECT `AREA NAME`, COUNT(*) AS `Count of Cases`
FROM {crime_df}
WHERE `AREA NAME` IS NOT NULL
GROUP BY `AREA NAME`
ORDER BY `Count of Cases` DESC
LIMIT 5
""", crime_df=df_pq)
result2_pq.collect()
end_pq = time.time()
diff_pq = end_pq-start_pq

start_csv = time.time()
result2_csv = spark.sql("""
SELECT `AREA NAME`, COUNT(*) AS `Count of Cases`
FROM {crime_df}
WHERE `AREA NAME` IS NOT NULL
GROUP BY `AREA NAME`
ORDER BY `Count of Cases` DESC
LIMIT 5
""", crime_df=df_csv)
result2_csv.collect()
end_csv = time.time()
diff_csv = end_csv-start_csv
print("Query Analytics for CSV vs Parquet Comparison\n")
print("CSV:",round(diff_csv,2),"\n")
print("Parquet:",round(diff_pq,2),"\n")
if diff_pq < diff_csv:
    print("Parquet is faster by:",round(diff_csv-diff_pq,2),"seconds\n")
else:
    print("CSV is faster by:",round(diff_pq-diff_csv,2),"seconds\n")
print("Which five LAPD areas have the most recorded incidents in this dataset?\n")
result2_csv

Query Analytics for CSV vs Parquet Comparison

CSV: 2.39 

Parquet: 1.16 

Parquet is faster by: 1.23 seconds

Which five LAPD areas have the most recorded incidents in this dataset?



AREA NAME,Count of Cases
Central,69668
77th Street,61756
Pacific,59513
Southwest,57434
Hollywood,52429


In [ ]:
#Cleaning rows
# - Wanted to change date format into yyyy-mm-dd (already changed by spark dateformat option)
# - Change timestamp to EST (12 hour clock)

#3.	How many recorded incidents occurred on each day, based on the records received so far?
start_pq = time.time()
result3_pq = spark.sql("""
SELECT * FROM
    (SELECT  `DATE OCC`, count(*) as `Count of Cases`
    FROM {crime_df}
    Where `DATE OCC` IS NOT NULL
    GROUP BY  `DATE OCC`)
   ORDER BY `DATE OCC` DESC
""", crime_df=df_pq)
result3_pq.collect()
end_pq = time.time()
diff_pq = end_pq-start_pq

start_csv = time.time()
result3_csv = spark.sql("""
SELECT * FROM
    (SELECT  `DATE OCC`, count(*) as `Count of Cases`
    FROM {crime_df}
    Where `DATE OCC` IS NOT NULL
    GROUP BY  `DATE OCC`)
   ORDER BY `DATE OCC` DESC
""", crime_df=df_csv)
result3_csv.collect()
end_csv = time.time()
diff_csv = end_csv-start_csv
print("Query Analytics for CSV vs Parquet Comparison\n")
print("CSV:",round(diff_csv,2),"\n")
print("Parquet:",round(diff_pq,2),"\n")
if diff_pq < diff_csv:
    print("Parquet is faster by:",round(diff_csv-diff_pq,2),"seconds\n")
else:
    print("CSV is faster by:",round(diff_pq-diff_csv,2),"seconds\n")
print("How many recorded incidents occurred on each day, based on the records received so far?\n")
result3_csv

Query Analytics for CSV vs Parquet Comparison

CSV: 2.06 

Parquet: 0.99 

Parquet is faster by: 1.06 seconds

How many recorded incidents occurred on each day, based on the records received so far?



DATE OCC,Count of Cases
2024-12-30,8
2024-12-29,59
2024-12-28,94
2024-12-27,80
2024-12-26,83
2024-12-25,50
2024-12-24,83
2024-12-23,98
2024-12-22,121
2024-12-21,124
